In [6]:
import pandas as pd
import numpy as np
from pathlib import Path
import pickle

# CONFIGURATION — All 11 activities

CONFIG = {
    # Sensors (wrist A+G + magnitude features)
    'sensor_columns': ['Ax_w', 'Ay_w', 'Az_w', 'Gx_w', 'Gy_w', 'Gz_w'],
    'magnitude_features': True,  # Add Acc_mag, Gyro_mag
    'num_features': 8,           # 6 raw + 2 magnitude

    # All 11 activities
    'selected_activities': [
        'knuckles_cracking',
        'hand_tapping',
        'sitting',
        'standing',
        'smoking',
        'nail_biting',
        'hair_pulling',
        'nape_rubbing',
        'hand_scratching',
        'forehead_rubbing',
        'ear_rubbing',
    ],

    'sampling_rate': 50,       # Hz
    'window_size': 600,        # Paper's best WS (12s — captures full activity cycles)
    'overlap': 0.9,            # 90% overlap (paper's best)
    'step_size': 60,           # 600 * (1 - 0.9) = 60

    # Leave-one-out: train on 9 users, test on user 3 (same as paper Table 5)
    'train_users': [2, 5, 6, 7, 8, 10, 14, 15, 16],
    'test_users': [3],
}

print("="*60)
print("DATA PREPROCESSING PIPELINE")
print("11 activities (WS=600, OP=90%, magnitude features)")
print("="*60)
print(f"\nConfiguration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

# STEP 1: Load Dataset
print("\n" + "="*60)
print("STEP 1: Loading Dataset")
print("="*60)

data_path = Path('../data/raw/AdamSense.csv')
df = pd.read_csv(data_path)
print(f"Loaded {len(df):,} samples")
print(f"Users in dataset: {sorted(df['User'].unique())}")

# STEP 2: Filter Selected Activities
print("\n" + "="*60)
print("STEP 2: Filtering Activities")
print("="*60)

df_filtered = df[df['Activity'].isin(CONFIG['selected_activities'])].copy()
print(f"Filtered to {len(df_filtered):,} samples")

print("\nSamples per activity:")
for activity in CONFIG['selected_activities']:
    count = len(df_filtered[df_filtered['Activity'] == activity])
    print(f"  {activity:20s}: {count:6,}")

# STEP 2.5: Add Magnitude Features
print("\n" + "="*60)
print("STEP 2.5: Adding Magnitude Features")
print("="*60)

df_filtered['Acc_mag'] = np.sqrt(
    df_filtered['Ax_w']**2 + df_filtered['Ay_w']**2 + df_filtered['Az_w']**2
)
df_filtered['Gyro_mag'] = np.sqrt(
    df_filtered['Gx_w']**2 + df_filtered['Gy_w']**2 + df_filtered['Gz_w']**2
)

sensor_cols = CONFIG['sensor_columns'] + ['Acc_mag', 'Gyro_mag']
print(f"Added magnitude features: Acc_mag, Gyro_mag")
print(f"Total features: {len(sensor_cols)} — {sensor_cols}")

# STEP 3: Create Activity Label Mapping
print("\n" + "="*60)
print("STEP 3: Creating Label Mapping")
print("="*60)

activity_to_label = {activity: idx for idx, activity in enumerate(CONFIG['selected_activities'])}
label_to_activity = {idx: activity for activity, idx in activity_to_label.items()}

print("Label mapping:")
for activity, label in activity_to_label.items():
    print(f"  {label}: {activity}")

df_filtered['label'] = df_filtered['Activity'].map(activity_to_label)

# STEP 4: Extract Sensor Data — Leave-One-Out
print("\n" + "="*60)
print("STEP 4: Extracting Sensor Data (leave-one-out)")
print("="*60)

def create_windows(data, labels, window_size, step_size):
    """Create sliding windows with majority-vote labeling."""
    windows = []
    window_labels = []

    for start in range(0, data.shape[0] - window_size + 1, step_size):
        end = start + window_size
        window_label_values = labels[start:end]
        unique, counts = np.unique(window_label_values, return_counts=True)

        if np.max(counts) / len(window_label_values) > 0.8:
            windows.append(data[start:end, :])
            window_labels.append(unique[np.argmax(counts)])

    return np.array(windows), np.array(window_labels)

all_train_windows = []
all_train_labels = []
all_test_windows = []
all_test_labels = []

for user in df_filtered['User'].unique():
    user_data = df_filtered[df_filtered['User'] == user]

    for activity in CONFIG['selected_activities']:
        activity_data = user_data[user_data['Activity'] == activity]

        if len(activity_data) < CONFIG['window_size']:
            continue

        X = activity_data[sensor_cols].values
        y = activity_data['label'].values

        X_windows, y_windows = create_windows(X, y, CONFIG['window_size'], CONFIG['step_size'])

        if len(X_windows) == 0:
            continue

        if user in CONFIG['train_users']:
            all_train_windows.append(X_windows)
            all_train_labels.append(y_windows)
        elif user in CONFIG['test_users']:
            all_test_windows.append(X_windows)
            all_test_labels.append(y_windows)

X_train = np.concatenate(all_train_windows, axis=0)
y_train = np.concatenate(all_train_labels, axis=0)
X_test = np.concatenate(all_test_windows, axis=0)
y_test = np.concatenate(all_test_labels, axis=0)

print(f"Created sliding windows (WS={CONFIG['window_size']}, overlap={CONFIG['overlap']*100:.0f}%)")
print(f"  Training set: {X_train.shape[0]:,} windows")
print(f"  Test set: {X_test.shape[0]:,} windows")
print(f"  Window shape: {X_train.shape[1:]} (samples, features)")

# STEP 5: Normalize Data (z-score, same as paper)
print("\n" + "="*60)
print("STEP 5: Normalizing Data (z-score)")
print("="*60)

mean = X_train.mean(axis=(0, 1))
std = X_train.std(axis=(0, 1))

print("Normalization parameters (per feature):")
print("Mean:", mean)
print("Std:", std)

X_train_normalized = (X_train - mean) / std
X_test_normalized = (X_test - mean) / std

print(f"\nData normalized")
print(f"  Train range: [{X_train_normalized.min():.2f}, {X_train_normalized.max():.2f}]")
print(f"  Test range: [{X_test_normalized.min():.2f}, {X_test_normalized.max():.2f}]")

# STEP 6: Verify Class Distribution
print("\n" + "="*60)
print("STEP 6: Class Distribution")
print("="*60)

print("Training set:")
unique_train, counts_train = np.unique(y_train, return_counts=True)
for label, count in zip(unique_train, counts_train):
    print(f"  {label}: {label_to_activity[label]:20s} - {count:6,} ({count/len(y_train)*100:5.2f}%)")

print("\nTest set:")
unique_test, counts_test = np.unique(y_test, return_counts=True)
for label, count in zip(unique_test, counts_test):
    print(f"  {label}: {label_to_activity[label]:20s} - {count:6,} ({count/len(y_test)*100:5.2f}%)")

# STEP 7: Save Preprocessed Data
print("\n" + "="*60)
print("STEP 7: Saving Preprocessed Data")
print("="*60)

output_dir = Path('../data/processed')
output_dir.mkdir(parents=True, exist_ok=True)

np.save(output_dir / 'X_train.npy', X_train_normalized)
np.save(output_dir / 'y_train.npy', y_train)
np.save(output_dir / 'X_test.npy', X_test_normalized)
np.save(output_dir / 'y_test.npy', y_test)

preprocessing_info = {
    'config': CONFIG,
    'mean': mean,
    'std': std,
    'activity_to_label': activity_to_label,
    'label_to_activity': label_to_activity,
    'train_shape': X_train_normalized.shape,
    'test_shape': X_test_normalized.shape,
}

with open(output_dir / 'preprocessing_info.pkl', 'wb') as f:
    pickle.dump(preprocessing_info, f)

print(f"Saved to {output_dir}/")
print(f"  - X_train.npy: {X_train_normalized.shape}")
print(f"  - y_train.npy: {y_train.shape}")
print(f"  - X_test.npy: {X_test_normalized.shape}")
print(f"  - y_test.npy: {y_test.shape}")
print(f"  - preprocessing_info.pkl")

print("\n" + "="*60)
print("PREPROCESSING COMPLETE!")
print("="*60)
print(f"\n  Activities: {len(CONFIG['selected_activities'])}")
print(f"  Features: {len(sensor_cols)} ({sensor_cols})")
print(f"  Split: leave-one-out (test user: {CONFIG['test_users']})")
print(f"  Window: {CONFIG['window_size']} samples ({CONFIG['window_size']/CONFIG['sampling_rate']:.1f}s), {CONFIG['overlap']*100:.0f}% overlap")
print(f"  Training: {X_train_normalized.shape[0]:,} windows")
print(f"  Test: {X_test_normalized.shape[0]:,} windows")

DATA PREPROCESSING PIPELINE
11 activities (WS=600, OP=90%, magnitude features)

Configuration:
  sensor_columns: ['Ax_w', 'Ay_w', 'Az_w', 'Gx_w', 'Gy_w', 'Gz_w']
  magnitude_features: True
  num_features: 8
  selected_activities: ['knuckles_cracking', 'hand_tapping', 'sitting', 'standing', 'smoking', 'nail_biting', 'hair_pulling', 'nape_rubbing', 'hand_scratching', 'forehead_rubbing', 'ear_rubbing']
  sampling_rate: 50
  window_size: 600
  overlap: 0.9
  step_size: 60
  train_users: [2, 5, 6, 7, 8, 10, 14, 15, 16]
  test_users: [3]

STEP 1: Loading Dataset
Loaded 709,582 samples
Users in dataset: [2, 3, 5, 6, 7, 8, 10, 14, 15, 16]

STEP 2: Filtering Activities
Filtered to 709,582 samples

Samples per activity:
  knuckles_cracking   : 67,498
  hand_tapping        : 63,220
  sitting             : 59,996
  standing            : 58,949
  smoking             : 66,937
  nail_biting         : 63,457
  hair_pulling        : 65,973
  nape_rubbing        : 67,859
  hand_scratching     : 64,490
 